# Debug Lift Charts: GLM vs GBM vs Final

**Purpose:** Compare lift charts and fit quality for:
- GLM predictions alone
- Final predictions (GLM × GBM_ratio)

**Use this to diagnose:**
- Is GLM already providing poor fit?
- Does GBM improve fit over GLM?
- Are deciles calculated correctly?

In [1]:
config_path = "config/car_liab/v1"

In [2]:
import matplotlib
import matplotlib.pyplot as plt

matplotlib.use("Agg")

import pandas as pd
import numpy as np
import yaml, sys, os
from pathlib import Path

# Find project root
cwd = Path.cwd()
for p in [cwd, cwd.parent, cwd.parent.parent]:
    if (p / "lib").exists():
        os.chdir(p)
        break

sys.path.insert(0, str(Path.cwd() / "lib"))
from debug_lift import debug_glm_vs_gbm, compare_lift_charts, plot_lift_comparison

print("#" * 60)
print("# DEBUG LIFT CHARTS")
print("#" * 60)
print(f"Working dir: {Path.cwd()}")

############################################################
# DEBUG LIFT CHARTS
############################################################
Working dir: /Users/Mach/dev/aps/code/26Dmodelv1


In [3]:
# Load config
pc_num = open("current.pc").read().strip()
pc_id = f"PC{pc_num}"

config_file = f"{config_path}/config.yaml"
with open(config_file, "r") as f:
    cfg = yaml.safe_load(f)

output_base = cfg["machines"][pc_id]["paths"]["output_path"]
target = cfg["experiment"]["target"]
exposure = cfg["experiment"]["exposure"]
target_method = cfg["model"].get("target_method", "direct")

print(f"\nOutput: {output_base}")
print(f"Target: {target}")
print(f"Exposure: {exposure}")
print(f"Target method: {target_method}")


Output: output/car_liab/v1
Target: pp_bi
Exposure: ee_bi_imps
Target method: residual


In [4]:
# Load train/test data with actuals
print(f"\n* Loading data...")
train_orig = pd.read_parquet(f"{output_base}/data/04b_train.parquet")
test_orig = pd.read_parquet(f"{output_base}/data/04b_test.parquet")

print(f"  Train: {train_orig.shape}")
print(f"  Test: {test_orig.shape}")


* Loading data...
  Train: (18709834, 122)
  Test: (7483698, 122)


In [5]:
# Load GLM predictions
print(f"\n* Loading GLM predictions...")
glm_train_df = pd.read_parquet(f"{output_base}/data/04c_train_glm.parquet")
glm_test_df = pd.read_parquet(f"{output_base}/data/04c_test_glm.parquet")

train_orig['glm_pred'] = glm_train_df['glm_pred'].values
test_orig['glm_pred'] = glm_test_df['glm_pred'].values

print(f"  Train GLM mean: {train_orig['glm_pred'].mean():.2f}")
print(f"  Test GLM mean: {test_orig['glm_pred'].mean():.2f}")


* Loading GLM predictions...
  Train GLM mean: 217.61
  Test GLM mean: 217.74


In [6]:
# Load final predictions (if available)
print(f"\n* Loading final predictions...")
try:
    pred_df = pd.read_parquet(f"{output_base}/results/05_predictions.parquet")
    
    # Merge predictions
    train_pred = pred_df[pred_df['fold'].isin([1,2,3,4,5,6,7])].copy()
    test_pred = pred_df[pred_df['fold'].isin([8,9,10])].copy()
    
    train_orig = train_orig.merge(train_pred[['pred']], left_index=True, right_index=True, how='left')
    test_orig = test_orig.merge(test_pred[['pred']], left_index=True, right_index=True, how='left')
    
    train_orig.rename(columns={'pred': 'final_pred'}, inplace=True)
    test_orig.rename(columns={'pred': 'final_pred'}, inplace=True)
    
    print(f"  Train final mean: {train_orig['final_pred'].mean():.2f}")
    print(f"  Test final mean: {test_orig['final_pred'].mean():.2f}")
    has_final = True
except:
    print("  Final predictions not found - will only compare GLM")
    has_final = False


* Loading final predictions...
  Final predictions not found - will only compare GLM


In [7]:
# Run comparison with charts
results = {}

for split_name, data in [('Train', train_orig), ('Test', test_orig)]:
    print(f'\n{"#"*60}')
    print(f'# {split_name} Set')
    print(f'{"#"*60}')
    
    split_results = compare_lift_charts(
        data=data,
        actual_col=target,
        weight_col=exposure,
        glm_col='glm_pred',
        final_col='final_pred' if has_final else None,
        bins=10,
        title_prefix=f'{split_name} - '
    )
    
    # Plot charts
    fig = plot_lift_comparison(split_results, figsize=(14, 5))
    if fig:
        fig.suptitle(f'{split_name} Lift Charts', y=1.02, fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.savefig(f'{output_base}/debug_lift_{split_name.lower()}.png', dpi=150, bbox_inches='tight')
        print(f'\n  Saved: debug_lift_{split_name.lower()}.png')
        plt.show()
    
    results[split_name] = split_results


############################################################
# Train Set
############################################################

Train - GLM Predictions
  Fit Quality:  0.5332
  Model Power:  0.2987
  Avg Error:    0.4668

Decile Summary:
 decile      weight  weight_pct     act    pred  act_rel  pred_rel  error
      1 1086105.868       9.500  58.390  69.315    0.411     0.332  0.187
      2 1143270.145      10.000  78.774 123.834    0.554     0.592  0.572
      3 1143270.140      10.000 104.243 153.597    0.733     0.735  0.473
      4 1143270.086      10.000 109.585 178.958    0.771     0.856  0.633
      5 1143269.244      10.000 136.023 200.323    0.957     0.958  0.473
      6 1143270.493      10.000 141.545 219.048    0.996     1.048  0.548
      7 1143269.960      10.000 170.871 238.463    1.202     1.141  0.396
      8 1143270.386      10.000 177.004 257.563    1.245     1.232  0.455
      9 1143269.203      10.000 190.637 281.581    1.341     1.347  0.477
     10 120043

/var/folders/v0/9dsqbvc54xb7m5hl9q0typp80000gn/T/ipykernel_16724/4175445122.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()



Test - GLM Predictions
  Fit Quality:  0.6317
  Model Power:  0.2990
  Avg Error:    0.3683

Decile Summary:
 decile     weight  weight_pct     act    pred  act_rel  pred_rel  error
      1 433931.628       9.500  61.970  69.148    0.406     0.331  0.116
      2 456769.967      10.000  75.558 123.754    0.496     0.592  0.638
      3 456769.943      10.000 124.235 153.510    0.815     0.734  0.236
      4 456770.885      10.000 149.784 178.900    0.983     0.856  0.194
      5 456770.239      10.000 152.147 200.316    0.998     0.958  0.317
      6 456770.221      10.000 142.385 219.091    0.934     1.048  0.539
      7 456769.936      10.000 165.802 238.546    1.088     1.141  0.439
      8 456770.494      10.000 196.904 257.669    1.292     1.233  0.309
      9 456769.692      10.000 182.826 281.674    1.199     1.348  0.541
     10 479609.643      10.500 262.824 353.366    1.724     1.691  0.345

  Saved: debug_lift_test.png


/var/folders/v0/9dsqbvc54xb7m5hl9q0typp80000gn/T/ipykernel_16724/4175445122.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
# Summary
print(f"\n{'='*60}")
print("SUMMARY")
print(f"{'='*60}")
for split in ['Train', 'Test']:
    if split in results:
        print(f"\n{split}:")
        for name, data in results[split].items():
            m = data['metrics']
            print(f"  {name:8s}: Fit={m['fit_quality']:.4f}, Power={m['model_power']:.4f}")


SUMMARY

Train:
  GLM     : Fit=0.5332, Power=0.2987

Test:
  GLM     : Fit=0.6317, Power=0.2990
